In [2]:
import sys
print(sys.version)

3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]


In [3]:
!python --version

Python 3.14.3


In [4]:
!py --version

Python 3.14.3


In [5]:
%pip --version

pip 25.3 from c:\Program Files\Python314\Lib\site-packages\pip (python 3.14)

Note: you may need to restart the kernel to use updated packages.


In [6]:
import sys
print(sys.executable)

c:\Program Files\Python314\python.exe


In [7]:
%pip install requests

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Step 1: Install Required Packages

In [8]:
%pip install requests beautifulsoup4 pandas sqlalchemy 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import sqlite3
import pandas as pd
import requests
from bs4 import BeautifulSoup
from sqlalchemy import create_engine

In [10]:
import sqlite3
print(sqlite3.sqlite_version)

3.50.4


In [11]:
from sqlalchemy import create_engine
import re

Select 3 Categories

In [12]:
categories = [
    "Travel",
    "Mystery",
    "Historical Fiction"
]

Step 6: Get Category URLs


In [13]:
BASE_URL = "http://books.toscrape.com/"

response = requests.get(BASE_URL)

soup = BeautifulSoup(response.text, "html.parser")

category_links = {}

for category in soup.select(".side_categories ul li ul li a"):
    name = category.text.strip()

    href = category["href"]

    category_links[name] = BASE_URL + href

category_links

{'Travel': 'http://books.toscrape.com/catalogue/category/books/travel_2/index.html',
 'Mystery': 'http://books.toscrape.com/catalogue/category/books/mystery_3/index.html',
 'Historical Fiction': 'http://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html',
 'Sequential Art': 'http://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html',
 'Classics': 'http://books.toscrape.com/catalogue/category/books/classics_6/index.html',
 'Philosophy': 'http://books.toscrape.com/catalogue/category/books/philosophy_7/index.html',
 'Romance': 'http://books.toscrape.com/catalogue/category/books/romance_8/index.html',
 'Womens Fiction': 'http://books.toscrape.com/catalogue/category/books/womens-fiction_9/index.html',
 'Fiction': 'http://books.toscrape.com/catalogue/category/books/fiction_10/index.html',
 'Childrens': 'http://books.toscrape.com/catalogue/category/books/childrens_11/index.html',
 'Religion': 'http://books.toscrape.com/catalogue/category/books/rel

Step 7: Build a Function to Scrape One Category

In [14]:
def scrape_category(category_name, category_url):
    
    books = []

    while category_url:
        
        response = requests.get(category_url)

        soup = BeautifulSoup(response.text, "html.parser")

        articles = soup.find_all("article", class_="product_pod")

        for article in articles:

            title = article.h3.a["title"]

            price = article.find(
                "p",
                class_="price_color"
            ).text.strip()

            rating_classes = article.find(
                "p",
                class_="star-rating"
            )["class"]

            star_rating = rating_classes[1]

            availability = article.find(
                "p",
                class_="instock availability"
            ).text.strip()

            books.append({
                "title": title,
                "price_gbp": price,
                "star_rating": star_rating,
                "availability": availability,
                "category": category_name
            })

        next_button = soup.select_one("li.next a")

        if next_button:
            next_page = next_button["href"]

            category_url = category_url.rsplit("/",1)[0] + "/" + next_page
        else:
            category_url = None

    return books

Step 8: Scrape Multiple Categories

In [15]:
selected_categories = [
    "Travel",
    "Mystery",
    "Historical Fiction"
]

all_books = []

for cat in selected_categories:

    print(f"Scraping {cat}...")

    data = scrape_category(
        cat,
        category_links[cat]
    )

    all_books.extend(data)

Scraping Travel...
Scraping Mystery...
Scraping Historical Fiction...


Step 9: Create DataFrame

In [16]:
df = pd.DataFrame(all_books)

df.head()

,title,price_gbp,star_rating,availability,category
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel


Convert Price to Float

In [17]:
df["price_gbp"] = (
    df["price_gbp"]
    .str.replace("£", "", regex=False)
)



In [18]:
df.head()

,title,price_gbp,star_rating,availability,category
0,It's Only the Himalayas,Â45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,Â48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â36.94,Two,In stock,Travel
4,Under the Tuscan Sun,Â37.33,Three,In stock,Travel


In [19]:
df["price_gbp"] = (
    df["price_gbp"]
    .str.replace("Â", "", regex=False)
)

In [20]:
df.shape

(69, 5)

In [21]:
df

,title,price_gbp,star_rating,availability,category
0,It's Only the Himalayas,45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,Two,In stock,Travel
4,Under the Tuscan Sun,37.33,Three,In stock,Travel
...,...,...,...,...,...
64,While You Were Mine,41.32,Five,In stock,Historical Fiction
65,The Secret Healer,34.56,Three,In stock,Historical Fiction
66,Starlark,25.83,Three,In stock,Historical Fiction
67,Lost Among the Living,27.70,Four,In stock,Historical Fiction


In [22]:
df["price_gbp"] = pd.to_numeric(
    df["price_gbp"],
    errors="coerce"
)

In [23]:
df.head()

,title,price_gbp,star_rating,availability,category
0,It's Only the Himalayas,45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,Two,In stock,Travel
4,Under the Tuscan Sun,37.33,Three,In stock,Travel


In [24]:
df.shape

(69, 5)

In [25]:
df.columns = df.columns.str.strip()
print(df.columns.tolist())

['title', 'price_gbp', 'star_rating', 'availability', 'category']


In [26]:
df['price_gbp'].isna().sum()

np.int64(0)

In [27]:
EUR_TO_INR = 110.7
df["price_inr"] = df["price_gbp"] * EUR_TO_INR

Convert Star Ratings to Integers

In [28]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["star_rating"].map(rating_map)

Convert Availability to Boolean

In [29]:
df["in_stock"] = df["availability"].str.contains(
    "In stock",
    case=False,
    na=False
)

Handle Parsing Errors Numeric Columns

In [30]:
median_price = df["price_gbp"].median()

df["price_gbp"] = df["price_gbp"].fillna(
    median_price
)

In [31]:
median_rating = round(
    df["rating"].median()
)

df["rating"] = df["rating"].fillna(
    median_rating
)

Non-Numeric Columns

In [32]:
df = df.dropna(
    subset=["title", "category"]
)

Numeric parsing failures are handled using median imputation because median is robust to outliers and prevents pipeline failure. Rows missing essential text fields such as title or category are dropped because those attributes cannot be reliably inferred.


Design Normalized Schema

Create SQLite Database

In [33]:
import sqlite3

conn = sqlite3.connect("books.db")

cursor = conn.cursor()

Create Tables

In [34]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY,
    title TEXT,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY(category_id)
    REFERENCES categories(category_id)
)
""")

Populate Categories

In [ ]:
# existing_categories = pd.read_sql(
#     "SELECT category_name FROM categories",
#     conn
# )

# new_categories = categories[
#     ~categories["category_name"].isin(existing_categories["category_name"])
# ]

# new_categories.to_sql(
#     "categories",
#     conn,
#     if_exists="append",
#     index=False
# )

TypeError: list indices must be integers or slices, not str

In [ ]:
categories_df.to_sql(
    "categories",
    conn,
    if_exists="append",
    index=False
)

DatabaseError: Execution failed

Add Category IDs

In [ ]:
cat_lookup = pd.read_sql(
    "SELECT * FROM categories",
    conn
)

In [ ]:
df = df.merge(
    cat_lookup,
    left_on="category",
    right_on="category_name"
)

In [ ]:
books_df = df[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category_id"
    ]
]

In [ ]:
df.head()

,title,price_gbp,star_rating,availability,category,price_inr,rating,in_stock,category_id_x,category_name_x,category_id_y,category_name_y
0,It's Only the Himalayas,45.17,Two,In stock,Travel,5000.319,2,True,3,Travel,3,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,Four,In stock,Travel,5471.901,4,True,3,Travel,3,Travel
2,See America: A Celebration of Our National Par...,48.87,Three,In stock,Travel,5409.909,3,True,3,Travel,3,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,Two,In stock,Travel,4089.258,2,True,3,Travel,3,Travel
4,Under the Tuscan Sun,37.33,Three,In stock,Travel,4132.431,3,True,3,Travel,3,Travel


In [36]:
# Create categories dataframe
categories_df = (
    df[['category_id_x', 'category_name_x']]
    .drop_duplicates()
    .rename(columns={
        'category_id_x': 'category_id',
        'category_name_x': 'category_name'
    })
)

print(categories_df.head())

KeyError: "None of [Index(['category_id_x', 'category_name_x'], dtype='str')] are in the [columns]"

In [ ]:
existing_ids = set(
    pd.read_sql(
        "SELECT category_id FROM categories",
        conn
    )['category_id']
)

new_categories = categories_df[
    ~categories_df['category_id'].isin(existing_ids)
]

new_categories.to_sql(
    'categories',
    conn,
    if_exists='append',
    index=False
)

0

In [37]:
books_df.to_sql(
    "books",
    conn,
    if_exists="append",
    index=False
)

NameError: name 'books_df' is not defined

Query 1: SELECT + WHERE

In [38]:
type(books_df)

NameError: name 'books_df' is not defined

In [39]:
query = """
SELECT *
FROM books
WHERE rating = 5;
"""

result = pd.read_sql(query, conn)
print(result)

    book_id                                              title  price_gbp  \
0        11                 1,000 Places to See Before You Die      26.08   
1        20             A Time of Torment (Charlie Parker #14)      48.35   
2        29  What Happened on Beale Street (Secrets of the ...      25.37   
3        30  The Bachelor Girl's Guide to Murder (Herringfo...      52.30   
4        34                  The Silkworm (Cormoran Strike #2)      23.05   
5        40                                  The Girl You Lost      12.29   
6        46            A Flight of Arrows (The Pathfinders #2)      55.53   
7        48                                       Mrs. Houdini      30.25   
8        57                              The Passion of Dolssa      28.32   
9        59                             Voyager (Outlander #3)      21.07   
10       60                                       The Red Tent      35.66   
11       64                             Between Shades of Gray      20.79   

In [40]:
cursor = conn.cursor()

cursor.execute("""
SELECT *
FROM books
WHERE rating = 5
""")

rows = cursor.fetchall()
print(rows)

[(11, '1,000 Places to See Before You Die', 26.08, 2887.056, 5, 1, 3), (20, 'A Time of Torment (Charlie Parker #14)', 48.35, 5352.345, 5, 1, 2), (29, 'What Happened on Beale Street (Secrets of the South Mysteries #2)', 25.37, 2808.4590000000003, 5, 1, 2), (30, "The Bachelor Girl's Guide to Murder (Herringford and Watts Mysteries #1)", 52.3, 5789.61, 5, 1, 2), (34, 'The Silkworm (Cormoran Strike #2)', 23.05, 2551.635, 5, 1, 2), (40, 'The Girl You Lost', 12.29, 1360.503, 5, 1, 2), (46, 'A Flight of Arrows (The Pathfinders #2)', 55.53, 6147.171, 5, 1, 1), (48, 'Mrs. Houdini', 30.25, 3348.675, 5, 1, 1), (57, 'The Passion of Dolssa', 28.32, 3135.024, 5, 1, 1), (59, 'Voyager (Outlander #3)', 21.07, 2332.449, 5, 1, 1), (60, 'The Red Tent', 35.66, 3947.562, 5, 1, 1), (64, 'Between Shades of Gray', 20.79, 2301.453, 5, 1, 1), (65, 'While You Were Mine', 41.32, 4574.124, 5, 1, 1), (69, "A Spy's Devotion (The Regency Spies of London #1)", 16.97, 1878.579, 5, 1, 1), (80, '1,000 Places to See Before

Query 1: SELECT + WHERE

In [41]:
query1 = """
SELECT *
FROM books
WHERE rating = 5;
"""

result = pd.read_sql(query1, conn)
print(result)

    book_id                                              title  price_gbp  \
0        11                 1,000 Places to See Before You Die      26.08   
1        20             A Time of Torment (Charlie Parker #14)      48.35   
2        29  What Happened on Beale Street (Secrets of the ...      25.37   
3        30  The Bachelor Girl's Guide to Murder (Herringfo...      52.30   
4        34                  The Silkworm (Cormoran Strike #2)      23.05   
5        40                                  The Girl You Lost      12.29   
6        46            A Flight of Arrows (The Pathfinders #2)      55.53   
7        48                                       Mrs. Houdini      30.25   
8        57                              The Passion of Dolssa      28.32   
9        59                             Voyager (Outlander #3)      21.07   
10       60                                       The Red Tent      35.66   
11       64                             Between Shades of Gray      20.79   

Query 2: ORDER BY

In [42]:
query2 = """
SELECT title, price_inr
FROM books
ORDER BY price_inr DESC;
"""

result = pd.read_sql(query2, conn)
print(result)

                                                 title  price_inr
0                        Boar Island (Anna Pigeon #19)   6584.436
1                        Boar Island (Anna Pigeon #19)   6584.436
2    The No. 1 Ladies' Detective Agency (No. 1 Ladi...   6387.390
3    The No. 1 Ladies' Detective Agency (No. 1 Ladi...   6387.390
4                     A Year in Provence (Provence #1)   6296.616
..                                                 ...        ...
133                                  The Girl You Lost   1360.503
134                         Hide Away (Eve Duncan #20)   1310.688
135                         Hide Away (Eve Duncan #20)   1310.688
136               Tastes Like Fear (DI Marnie Rome #3)   1183.383
137               Tastes Like Fear (DI Marnie Rome #3)   1183.383

[138 rows x 2 columns]


Query 3: LIMIT

In [43]:
query3 = """
SELECT title, price_inr
FROM books
ORDER BY price_inr DESC
LIMIT 10;
"""

result = pd.read_sql(query3, conn)
print(result)

                                               title  price_inr
0                      Boar Island (Anna Pigeon #19)   6584.436
1                      Boar Island (Anna Pigeon #19)   6584.436
2  The No. 1 Ladies' Detective Agency (No. 1 Ladi...   6387.390
3  The No. 1 Ladies' Detective Agency (No. 1 Ladi...   6387.390
4                   A Year in Provence (Provence #1)   6296.616
5                   A Year in Provence (Provence #1)   6296.616
6                                The Past Never Ends   6254.550
7                                The Past Never Ends   6254.550
8                   The Last Painting of Sara de Vos   6149.385
9                   The Last Painting of Sara de Vos   6149.385


Query 4: DISTINCT

In [44]:
query4 = """
SELECT DISTINCT rating
FROM books
ORDER BY rating;
"""

result = pd.read_sql(query4, conn)
print(result)

   rating
0       1
1       2
2       3
3       4
4       5


Query 5: BETWEEN

In [45]:
query5 = """
SELECT *
FROM books
WHERE price_gbp BETWEEN 20 AND 40;
"""

result = pd.read_sql(query5, conn)
print(result)

    book_id                                              title  price_gbp  \
0         4  Vagabonding: An Uncommon Guide to the Art of L...      36.94   
1         5                               Under the Tuscan Sun      37.33   
2         7                           The Great Railway Bazaar      30.54   
3         9  The Road to Little Dribbling: Adventures of an...      23.21   
4        10          Neither Here nor There: Travels in Europe      38.95   
..      ...                                                ...        ...   
61      129                                       The Red Tent      35.66   
62      133                             Between Shades of Gray      20.79   
63      135                                  The Secret Healer      34.56   
64      136                                           Starlark      25.83   
65      137                              Lost Among the Living      27.70   

    price_inr  rating  in_stock  category_id  
0    4089.258       2       

Query 6: JOIN (Important)

In [46]:
query5 = """
SELECT *
FROM categories
"""

result = pd.read_sql(query5, conn)
print(result)

   category_id       category_name
0            1  Historical Fiction
1            2             Mystery
2            3              Travel


In [47]:
query6 = """
SELECT
    b.title,
    b.rating,
    c.category_name
FROM books b
JOIN categories c
    ON b.category_id = c.category_id
ORDER BY b.rating DESC
LIMIT 10;
"""

result = pd.read_sql(query6, conn)
print(result)

                                               title  rating  \
0                 1,000 Places to See Before You Die       5   
1             A Time of Torment (Charlie Parker #14)       5   
2  What Happened on Beale Street (Secrets of the ...       5   
3  The Bachelor Girl's Guide to Murder (Herringfo...       5   
4                  The Silkworm (Cormoran Strike #2)       5   
5                                  The Girl You Lost       5   
6            A Flight of Arrows (The Pathfinders #2)       5   
7                                       Mrs. Houdini       5   
8                              The Passion of Dolssa       5   
9                             Voyager (Outlander #3)       5   

        category_name  
0              Travel  
1             Mystery  
2             Mystery  
3             Mystery  
4             Mystery  
5             Mystery  
6  Historical Fiction  
7  Historical Fiction  
8  Historical Fiction  
9  Historical Fiction  


Read at Least Two Queries into Pandas

In [48]:
top_books_df = pd.read_sql(query3, conn)

rating_df = pd.read_sql(query4, conn)

Reproduce JOIN Using Pandas Merge

In [49]:
query7 = """
SELECT
b.title,
b.rating,
c.category_name
FROM books b
JOIN categories c
ON b.category_id = c.category_id
"""

result = pd.read_sql(query7, conn)
print(result)

                                                 title  rating  \
0                              It's Only the Himalayas       2   
1    Full Moon over Noahâs Ark: An Odyssey to Mou...       4   
2    See America: A Celebration of Our National Par...       3   
3    Vagabonding: An Uncommon Guide to the Art of L...       2   
4                                 Under the Tuscan Sun       3   
..                                                 ...     ...   
133                                While You Were Mine       5   
134                                  The Secret Healer       3   
135                                           Starlark       3   
136                              Lost Among the Living       4   
137  A Spy's Devotion (The Regency Spies of London #1)       5   

          category_name  
0                Travel  
1                Travel  
2                Travel  
3                Travel  
4                Travel  
..                  ...  
133  Historical Fiction  

In [50]:
books_table = pd.read_sql(
    "SELECT * FROM books",
    conn
)

categories_table = pd.read_sql(
    "SELECT * FROM categories",
    conn
)

In [51]:
merged_df = pd.merge(
    books_table,
    categories_table,
    on="category_id",
    how="inner"
)

In [52]:
merged_df[
    [
        "title",
        "rating",
        "category_name"
    ]
]

,title,rating,category_name
0,It's Only the Himalayas,2,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,4,Travel
2,See America: A Celebration of Our National Par...,3,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,2,Travel
4,Under the Tuscan Sun,3,Travel
...,...,...,...
133,While You Were Mine,5,Historical Fiction
134,The Secret Healer,3,Historical Fiction
135,Starlark,3,Historical Fiction
136,Lost Among the Living,4,Historical Fiction
